# v9 — FP8 E4M3 KV cache — the gate (CUDA-core, Colab T4)

Forks v8.7 (score-stationary GQA M-packing) and changes **one variable: KV storage precision** —
the paged K/V pool holds FP8 E4M3 (1 byte) instead of FP16 (2), dequantized per-tile at the smem
gather. The score-stationary inner loop is byte-identical, so `v9_fp8` vs `v8_gqa_ss` is a clean
byte-only A/B.

**Roofline:** decode AI = 2G/b rises 8.0 → 16.0 (b: 2 → 1); the HBM floor **halves**; limiter stays
HBM. The model is **blind** to dequant latency and to the L2-residency confound.

**Prediction (record before run):** on the L2-resident micro-bench, FP8 is likely **capacity-only**
(no µs/tok win). **Counter:** if still per-CTA-bound even past L2, FP8 is capacity-only on this kernel
and the residual ceiling is launch/per-CTA (→ persistent-kernel territory, a future lever).

**Deliverables:** (1) correctness vs the E4M3 apples-to-apples oracle; (2) the E4M3 **quantization
RMSE** vs fp16 KV; (3) µs/tok + %HBM A/B vs v8.7; (4) reclaim-at-batch. Build/correctness/bench here;
the bandwidth *verdict* (locked-clock, L2-flush, past-L2 sweep) is v9 Task 1 on a root T4, out of scope.

## 0. Dependencies + GPU (venv-safe)

In [ ]:
import os, sys, subprocess

def pip(*pkgs, extra=()):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *pkgs, *extra], check=True)

# 1) Physical GPU on this runtime? (Colab defaults to CPU; pick a GPU explicitly.)
try:
    has_gpu = subprocess.run(['nvidia-smi'], capture_output=True).returncode == 0
except FileNotFoundError:
    has_gpu = False
if not has_gpu:
    raise SystemExit(
        'No GPU on this Colab runtime. FIX: Runtime > Change runtime type > T4 GPU > Save, '
        'then Runtime > Restart session, then re-run from the top. (This kernel needs a Turing T4.)')

# 2) Install deps (incl. numpy) BEFORE importing torch, so torch's numpy bridge initializes.
pip('ninja', 'pytest', 'numpy')

# 3) torch present AND CUDA-enabled? A CPU-only wheel raises "not compiled with CUDA" on any kernel.
try:
    import torch
    cuda_ok = torch.cuda.is_available()
except ImportError:
    torch, cuda_ok = None, False

if not cuda_ok:
    pip('torch', extra=('--index-url', 'https://download.pytorch.org/whl/cu124'))
    raise SystemExit(
        'A GPU is present but torch was a CPU-only build -- installed the CUDA build. NOW: '
        'restart the kernel/session, then re-run this cell.')

# vast.ai/venv: !-cells spawn a bare shell without the venv on PATH -> `python` not found.
os.environ['PATH'] = os.path.dirname(sys.executable) + os.pathsep + os.environ.get('PATH', '')

# torch.float8_e4m3fn must exist (>=2.1) — v9 stores the KV cache as E4M3 bytes.
assert hasattr(torch, 'float8_e4m3fn'), 'this torch lacks float8_e4m3fn; upgrade torch (>=2.1)'
print('torch', torch.__version__, '| cuda', torch.version.cuda, '| cap', torch.cuda.get_device_capability())
!nvidia-smi --query-gpu=name,compute_cap --format=csv

## 1. Get the repo

In [ ]:
REPO_URL = 'https://github.com/gkienpham-cmd/flashattention-cuda.git'  # public; plain clone works
import os, sys, subprocess
if os.path.basename(os.getcwd()) != 'flashattention-cuda':
    if not os.path.isdir('flashattention-cuda'):
        subprocess.run(['git', 'clone', REPO_URL], check=True)
    os.chdir('flashattention-cuda')
subprocess.run(['git', 'pull', 'origin', 'main'])
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())
print('cwd', os.getcwd())

## 2. Roofline — FP8 doubles AI, halves the HBM floor (still BLIND)

In [ ]:
from roofline.archs import get_arch
from roofline.model import estimate
arch = get_arch('sm_75')
print('arch:', arch.name, '| HBM', arch.hbm_bw_gbps, 'GB/s')
# Decode AI = 2G/b (b = bytes/elem): fp16 (b=2) -> AI=G; fp8 (b=1) -> AI=2G. At G=8: fp16=8.0, fp8=16.0.
# Print both precisions side by side to show FP8 doubles AI and halves the HBM floor.
print(f"{'G':>3} | {'AI fp16':>8} | {'AI fp8':>8} | {'limiter':>7} | {'t_hbm fp16':>11} | {'t_hbm fp8':>10}")
for G in (1,2,4,8,16,32):
    e16 = estimate(arch, B=8, H=8, N_q=1, N_k=8192, d=128, precision='fp16', G=G)
    e8  = estimate(arch, B=8, H=8, N_q=1, N_k=8192, d=128, precision='fp8',  G=G)
    print(f'{G:>3} | {e16.arithmetic_intensity:8.1f} | {e8.arithmetic_intensity:8.1f} | '
          f'{e8.limiter.upper():>7} | {e16.t_hbm*1e3:8.4f}ms | {e8.t_hbm*1e3:7.4f}ms')
print('\nFP8 DOUBLES AI (b: 2->1) and HALVES the HBM floor; limiter stays HBM (16 << T4 fp16 ridge 203).')
print('The roofline is BLIND to dequant latency AND to the L2-residency confound.')
print('PREDICTION: on the L2-resident micro-bench, FP8 is likely capacity-only (no us/tok win);')
print('COUNTER: if still per-CTA-bound past L2, residual ceiling is launch/per-CTA, not bytes.')

## 3. Build v9_fp8 (JIT) — WATCH for E4M3 ptxas failure on sm_75 (int8 fallback if so)

In [ ]:
import glob, os, shutil
for d in glob.glob(os.path.expanduser('~/.cache/torch_extensions/*/fa_v9_fp8')):
    if not glob.glob(os.path.join(d, '*.so')):
        shutil.rmtree(d, ignore_errors=True); print('cleaned stale build:', d)
from bindings.load import build_kernel
# WATCH the ptxas output: if E4M3 (__nv_cvt_fp8_to_halfraw) fails to compile on sm_75, flip
# dequant_e4m3 in kernels/v9_fp8/fp8_attention.cu to the int8-symmetric fallback (1-line swap)
# and update build_paged_kv_fp8/quantize_fp8_e4m3 to match, then rebuild.
fp8 = build_kernel('v9_fp8'); print('built v9 (FP8 E4M3 KV):', fp8)

## 4. Correctness gate — v9_fp8 + v8.7 regression (Gate 1 of 2)

In [ ]:
!python -m pytest tests/test_correctness.py -k "v9_fp8 or v8_gqa_ss" -q

## 5. Accuracy — FP8 E4M3 quantization RMSE vs fp16 KV (the deliverable)

In [ ]:
# Accuracy deliverable: FP8 E4M3 KV quantization error. Two numbers per shape:
#   (1) kernel vs apples-to-apples oracle (SDPA on the SAME dequantized E4M3 bytes) -> should be ~0
#       up to the FP16-accum band (this is what the correctness gate asserts at 5e-2);
#   (2) kernel vs the ORIGINAL fp16 KV (SDPA, un-quantized) -> the QUANTIZATION RMSE, the real number.
import torch
from fa_kernels import fp8_attention
from fa_kernels.paged import build_paged_kv_fp8
from fa_kernels.reference import sdpa_reference_gqa, sdpa_reference_gqa_fp8

def err(a, b):
    rmse = (a - b).pow(2).mean().sqrt().item()
    maxabs = (a - b).abs().max().item()
    return rmse, maxabs

print(f"{'shape':>16} | {'vs E4M3 oracle (rmse/max)':>26} | {'vs fp16 KV (rmse/max)':>24}")
for d in (64, 128):
    for G in (1, 8):
        torch.manual_seed(9)
        B, H_kv, N_k = 1, 2, 8192
        H_q = G * H_kv
        ps = 128
        q = torch.randn(B, H_q,  1,   d, device='cuda')
        k = torch.randn(B, H_kv, N_k, d, device='cuda')
        v = torch.randn(B, H_kv, N_k, d, device='cuda')
        kp, vp, bt, nk, sk, sv = build_paged_kv_fp8(k, v, ps, seed=9)
        out = fp8_attention(q, kp, vp, bt, ps, nk, sk, sv, causal=False, q_offset=0)
        r1, m1 = err(out, sdpa_reference_gqa_fp8(q, k, v, sk, sv, causal=False))   # apples-to-apples
        r2, m2 = err(out, sdpa_reference_gqa(q, k, v, causal=False))               # quantization error
        print(f"{f'{B}x{H_q}x1x{d}/{N_k} G{G}':>16} | {r1:11.2e}/{m1:11.2e} | {r2:10.2e}/{m2:10.2e}")
print('\n(1) should sit in the FP16 band; (2) is the E4M3 quantization RMSE = the accuracy deliverable.')

## 6. THE A/B — v9_fp8 vs v8.7 (byte-isolated), G-sweep

In [ ]:
print('=== v8.7 baseline: v8_gqa_ss (FP16 KV, score-stationary) ===')
!python -m bench.harness --backend v8_gqa_ss --decode --seq 8192 --heads 32 --gqa-group 1 2 4 8 16 32
print('\n=== v9: v9_fp8 (FP8 E4M3 KV, same score-stationary loop) ===')
# In the v9 rows, "vs naive" = the FP8-vs-FP16(v8.7) byte-isolation ratio (same packing); a trailing
# "L2!" on %HBM means the KV streamed from L2 (effective_bw > HBM peak) -> %HBM is NOT a boundedness signal.
!python -m bench.harness --backend v9_fp8 --decode --seq 8192 --heads 32 --gqa-group 1 2 4 8 16 32

## 7. Reclaim-at-batch (G=8) — does halving KV bytes move µs/tok at B≥8?

In [ ]:
print('=== v8.7: v8_gqa_ss (FP16 KV) ===')
!python -m bench.harness --backend v8_gqa_ss --decode --seq 8192 --heads 8 --gqa-group 8 --batch-sweep 1 8 16 32 64
print('\n=== v9: v9_fp8 (FP8 KV) ===')
!python -m bench.harness --backend v9_fp8 --decode --seq 8192 --heads 8 --gqa-group 8 --batch-sweep 1 8 16 32 64